# Load Dim Customer Table

In [0]:
# ci.customer_id -> internal surrogate key for crm_customer_information table,
# ci.customer_key -> business key (aw0001xxxx format)
# de.customer_key -> business key (aw0001xxxx format) from erp_customer_demographic table


query = """
SELECT
    --  Identifiers
    -- Surrogate key — works for CRM-only, ERP-only, both
    ROW_NUMBER() OVER (ORDER BY COALESCE(ci.customer_key, de.customer_key)) AS customer_key,
    COALESCE(ci.customer_key, de.customer_key) AS customer_nk,  -- business key (aw0001xxxx format)

    ci.customer_id,      -- CRM internal surrogate (NULL on ERP-only rows)

    -- personal details
    ci.customer_firstname AS first_name,
    ci.customer_lastname  AS last_name,
    ci.customer_marital_status AS marital_status,

    -- demographics
    la.country AS country,
    -- GENDER RECONCILIATION
    CASE
        -- Both have real gender values
        WHEN COALESCE(ci.customer_gender,'n/a') <> 'n/a' AND COALESCE(de.gender,'n/a') <> 'n/a'
            THEN
                CASE
                    WHEN ci.customer_gender = de.gender
                        THEN ci.customer_gender      -- if they match then select gender from CRM
                    ELSE 'conflicting'
                END

        -- Only CRM has it
        WHEN COALESCE(ci.customer_gender,'n/a') <> 'n/a' THEN ci.customer_gender

        -- Only ERP has it
        WHEN COALESCE(de.gender,'n/a') <> 'n/a' THEN de.gender

        -- Both missing
        ELSE 'unknown'
    END AS gender,

    -- Source tracking column — for later quality checks
    CASE
        WHEN COALESCE(ci.customer_gender,'n/a') <> 'n/a' AND COALESCE(de.gender,'n/a') <> 'n/a' THEN
            CASE
                WHEN ci.customer_gender = de.gender 
                    THEN 'both - agree'
                    ELSE 'both - conflict'
            END
        WHEN COALESCE(ci.customer_gender, 'n/a') <> 'n/a' THEN 'crm only'
        WHEN COALESCE(de.gender,   'n/a') <> 'n/a' THEN 'erp only'
        ELSE 'missing in both'
    END AS gender_source,

    -- Other core attributes
    de.dob AS birth_date,
    ci.customer_create_date AS create_date,

    CURRENT_DATE AS load_date

FROM bike_data.silver.crm_customer_information ci
FULL OUTER JOIN bike_data.silver.erp_customer_demographics de
    ON ci.customer_key = de.customer_key
LEFT JOIN bike_data.silver.erp_customer_location la
    ON COALESCE(ci.customer_key, de.customer_key) = la.customer_key

WHERE COALESCE(ci.customer_key, de.customer_key) IS NOT NULL AND ci.customer_id IS NOT NULL
"""
df = spark.sql(query)
display(df)

# Write to Gold Table

In [0]:
(
  df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bike_data.gold.dim_customer")
)